In [2]:
using HDF5
using Unitful
import PhysicalConstants.CODATA2018: m_n, e, ħ
using FileIO
using GeometryBasics
using LinearAlgebra
using BenchmarkTools

In [3]:
# Defining a function to calculate the magnitude of the wavevector, in Angstrom^-1, of the neutron from its energy.

function k_calc(E)
    """
    Calculates the magnitude of the wavevector, in Angstrom^-1, from an energy, in meV.

    Parameters
    ----------
    E (float): Energy, in meV.

    Returns
    -------
    mag_k (float): Magnitude of wavevector, in Angstrom^-1.
    """
    mag_k = sqrt(2 * m_n * E * e * (1e-3)) / (ħ * (1e10))
    return ustrip(mag_k)
end


k_calc (generic function with 1 method)

In [4]:
# Extracting the contents of the .nxspe file.

Ei, azi, pol, data, delE = h5open("LET104215_3.7meV_1to1.nxspe", "r") do f
    # Initial energy in meV.
    Ei = read(f["ws_out/NXSPE_info/fixed_energy"])[1]
    # Azimuthal angles in degrees.
    azi = read(f["ws_out/data/azimuthal"])
    # Polar angles in degrees.
    pol = read(f["ws_out/data/polar"])
    # Measured signal for each energy bin and each detector.
    data = read(f["ws_out/data/data"])
    # Energy bin centres, in meV, equal to energy change of neutron.
    delE = read(f["ws_out/data/energy"])
    return Ei, azi, pol, data, delE
end

(3.7, [-137.16071701049805, -137.39026260375977, -137.62151336669922, -137.8544692993164, -138.08910751342773, -138.32544326782227, -138.56351470947266, -138.80327224731445, -139.0447883605957, -139.2881088256836  …  41.245439529418945, 41.48468208312988, 41.7221097946167, 41.9577579498291, 42.19169521331787, 42.42388725280762, 42.65436553955078, 42.883137702941895, 43.110212326049805, 43.33559226989746], [48.28250598907471, 48.177175521850586, 48.07186985015869, 47.9666051864624, 47.8613977432251, 47.75627422332764, 47.651217460632324, 47.546268463134766, 47.44140434265137, 47.33662223815918  …  131.6914520263672, 131.5868377685547, 131.4821891784668, 131.37750625610352, 131.27277374267578, 131.16801834106445, 131.06324005126953, 130.95844650268555, 130.8536720275879, 130.74888229370117], [NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … NaN NaN; NaN NaN … NaN NaN], [-2.9600000000000004, -2.9415000000000004, -2.9230000000000005, -2.9045000000000005, -2.8860000000000006, -2.86750000

In [5]:
# Calculating Ef, ki, N_bins, N_detectors from the extracted content of the .nxspe file.


# Finding the initial wavevector, in Angstrom^-1, from the initial energy.
ki = zeros(3)
ki[1] = k_calc(Ei)

# Extracting the number of energy bins and number of detectors.
const N_bins = length(delE) - 1
const N_detectors = length(azi)

# Finding the bin centres by averaging the energies on each end of the bin.
E_bins = zeros(N_bins)
for i in 1:N_bins
    E_bins[i] = (delE[i] + delE[i+1]) / 2
end

# Determining the final neutron energy, Ef, based on which energy bin we are considering.
Ei_bins = Ei * ones(N_bins)
Ef_bins = Ei_bins - E_bins

320-element Vector{Float64}:
 6.65075
 6.632250000000001
 6.6137500000000005
 6.595250000000001
 6.5767500000000005
 6.558250000000001
 6.539750000000001
 6.521250000000001
 6.502750000000001
 6.484250000000001
 ⋮
 0.8972500000000094
 0.8787500000000099
 0.8602500000000095
 0.84175000000001
 0.8232500000000096
 0.8047500000000101
 0.7862500000000097
 0.7677500000000101
 0.7492500000000049

In [6]:
# Calculating the final wavevector, in Angstrom^-1, for each energy bin and each detector.


mag_kf = k_calc.(Ef_bins)
# Reshaping the arrays to allow for broadcasting.
pol_col = reshape(pol, :, 1)
azi_col = reshape(azi, :, 1)
mag_kf_row = reshape(mag_kf, 1, :)
# Determing the components of the final wavevector using broadcasting
kx = mag_kf_row .* (sin.(deg2rad.(pol_col)) .* cos.(deg2rad.(azi_col)))
ky = mag_kf_row .* (sin.(deg2rad.(pol_col)) .* sin.(deg2rad.(azi_col)))
kz = mag_kf_row .* cos.(deg2rad.(pol_col))
# Reshaping these final wavevector grids to align with the grid of data.
kx = transpose(kx)
ky = transpose(ky)
kz = transpose(kz)


320×98304 transpose(::Matrix{Float64}) with eltype Float64:
 1.1922    1.19465   1.19711   1.19955   …  -1.17438   -1.1719    -1.16942
 1.19054   1.19299   1.19544   1.19788      -1.17274   -1.17027   -1.16779
 1.18888   1.19133   1.19377   1.19621      -1.17111   -1.16864   -1.16616
 1.18721   1.18966   1.1921    1.19454      -1.16947   -1.167     -1.16453
 1.18555   1.18799   1.19043   1.19286      -1.16783   -1.16536   -1.1629
 1.18388   1.18632   1.18875   1.19118   …  -1.16618   -1.16372   -1.16126
 1.18221   1.18464   1.18707   1.1895       -1.16454   -1.16208   -1.15962
 1.18053   1.18297   1.18539   1.18782      -1.16289   -1.16044   -1.15798
 1.17886   1.18129   1.18371   1.18613      -1.16124   -1.15879   -1.15634
 1.17718   1.17961   1.18203   1.18444      -1.15958   -1.15714   -1.15469
 ⋮                                       ⋱                        
 0.437895  0.438797  0.439697  0.440596     -0.431349  -0.43044   -0.429529
 0.433357  0.43425   0.435141  0.43603      -0.4

In [7]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.


stl = load("crystal.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const N_faces = length(indices)

2142

In [8]:
# Calculating and storing the vectors parallel to each face, E2 = V2 - V1 and E3 = V3 - V1, in preparation for hte Moller-Trumbore Algorithm.
# The vertices of the triangular faces are labelled V1, V2, V3.

E2s = vertices[getindex.(indices, 2)] - vertices[getindex.(indices, 1)]
E3s = vertices[getindex.(indices, 3)] - vertices[getindex.(indices, 1)]

2142-element Vector{Point{3, Float32}}:
 [0.20006466, -0.12500381, 0.021842957]
 [-0.086564064, -0.24712181, 0.03823471]
 [-0.13139248, 0.22102165, -0.017211914]
 [0.11132717, -0.22167778, 0.014579773]
 [-0.1242342, -0.2677498, 0.004627228]
 [0.13139248, -0.22102165, 0.017211914]
 [-0.22294426, 0.20596504, 0.0055160522]
 [0.24161053, 0.0130290985, 0.010730743]
 [-0.16730404, 0.17181778, 0.01745224]
 [-0.32619762, -0.06916809, -0.0008392334]
 ⋮
 [0.28863525, -0.07749939, -0.0007972717]
 [-0.2492981, 0.23089218, -0.0039901733]
 [0.11425209, -0.24721527, 0.021465302]
 [-0.31781864, 0.21372223, -0.01871872]
 [0.26812553, -0.19742012, 0.006767273]
 [-0.03742695, 0.2215786, -0.019702911]
 [-0.28863525, 0.07749939, 0.0007972717]
 [-0.26812553, 0.19742012, -0.006767273]
 [0.050290108, 0.3967991, -0.016407013]

In [9]:
# Determining the coordinates of the points within our sample used for the Monte Carlo approximation of the volume integral.

const N_MC = 1
MC_coords = zeros(N_MC, 3)
# For this test, only one sample point used (at midpoint of coordinates).
max_coord = [maximum(getindex.(vertices, 1)), maximum(getindex.(vertices, 2)), maximum(getindex.(vertices, 3))]
min_coord = [minimum(getindex.(vertices, 1)), minimum(getindex.(vertices, 2)), minimum(getindex.(vertices, 3))]
MC_coords[1, :] = (min_coord + max_coord) / 2

3-element Vector{Float32}:
 16.6475
 20.065285
 47.947563

In [10]:
# Setting the (estimated) parameters of the sample.

# The number density of the sample in cm^-3.
const n = 1e23
# The reference absorption cross section at 25.3 meV in cm^2.
const axs_ref = 1e-23
const E_ref = 25.3

25.3

In [11]:
# Defining the function that calculates the absorption cross sections for the inputted energy.

function axs(E, axs_ref, E_ref)
    """
    Determines the absorption cross section (axs) for the inputted energy based on the absorption of the sample at a known, reference energy.

    Parameters
    ----------
    E (float): Energy in meV.
    axs_ref (float): Absorption cross section at E_ref in cm^2.
    E_ref (float): Reference energy in meV.

    Returns
    -------
    axs (float): Absorption cross section in cm^2.
    """
    return axs_ref * sqrt(E_ref / E)
end
@benchmark axs(Ei, axs_ref, E_ref)

BenchmarkTools.Trial: 10000 samples with 996 evaluations per sample.
 Range (min … max):  27.510 ns … 862.349 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     37.751 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   39.793 ns ±  13.056 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▃     ▃ ▄▂█▇▅▄▄▃▂▂▁                                          ▂
  █▆▆▁▁▄█████████████▇▇▅▆▆▅▆██▇▇███▇▇▇▆▆▆▅▄▅▅▄▅▅▆▆▁▅▅▅▆▅▅▅▅▄▅▄ █
  27.5 ns       Histogram: log(frequency) by time      83.8 ns <

 Memory estimate: 16 bytes, allocs estimate: 1.

In [12]:
# Calculating the pre-scattering absorption cross section.

const axsi = axs(Ei, axs_ref, E_ref)

2.614925971770107e-23

In [16]:
# Creating the function to determine the length of the paths the neutrons take within the sample.
# Broadcasted

function path_len(E2s, E3s, D, Ps, dets, origin, vertices, indices)
    """
    Calculates the distance between a given point in the sample (origin) and a triangular face of the mesh that describes the surface. 
    Iterates through each face to determine which one is intersected.
    Exploits the method described in 'Fast, Minimum Storage Ray-Triangle Intersection' by Moller and Trumbore.

    Parameters
    ----------
    E2s (N_faces x 3 array of floats): Array containing vectors parallel to each face, equal to V2 - V1.
    E3s (N_faces x 3 array of floats): Array containing vectors parallel to each face, equal to V3 - V1.
    D (3-dimensional array of floats): Direction vector.
    Ps (N_faces x 3 array of floats): Array containing P = D x E3 for each face.
    dets (N_faces - dimensional array of floats): Array containing det = P.E2 = (D x E3).E2 for each face.
    origin (3-dimensional array of floats): Coordinates of scattering sites.
    vertices (N_faces x 3 array of floats): Vertices of triangular faces.
    indices (N_faces x 3 array of floats): Indices describing which vertices form which triangles.

    Returns
    -------
    path_length (float): Distance between origin and the surface the neutron path intersects, in units of the .stl file.
    """
    # If det = P.E2 = (D x E3).E2 = 0, the path is parallel to the triangular face, so it can never intersect it.
    # Keeping only positive determinants, equivalent to triangular faces in the forward direction.
    idx = findall(dets .> 1e-10)
    # Calculating T = origin - V1 and Q = T x E2 required for the MT algorithm.
    T = fill(origin, length(idx)) - vertices[getindex.(indices[idx], 1)]
    Q = cross.(T, E2s[idx])
    # Calculating the barycentric coordinates, (u,v), of the intersection.
    u = (1 ./ dets[idx]) .* (dot.(Ps[idx], T))
    v = (1 ./ dets[idx]) .* (dot.(Q, fill(D, length(idx))))
    # Determining whether the intersection point lies within the triangle.
    for i in findall((v .>= 0) .&& (u .>= 0) .&& (u + v .<= 1))
        t = (1 / dets[idx[i]]) * dot(Q[i], E3s[idx[i]])
        # Determining the path length based on t and the magnitude of the inputted direction vector.
        path_length = abs(t) * norm(D)
        return path_length
    end
end
@benchmark path_len(E2s, E3s, -ki, cross.(fill(-ki, N_faces), E3s), dot.(Pi, E2s), MC_coords[1,:], vertices, indices)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   96.300 μs … 136.369 ms  ┊ GC (min … max):  0.00% … 99.66%
 Time  (median):     195.300 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   251.180 μs ±   1.631 ms  ┊ GC (mean ± σ):  17.96% ±  4.58%

  ▅█▂                                                            
  ███▆▄▃▂▂▂▂▂▂▁▂▂▅▅▃▂▃▂▂▂▃▄▅▃▂▂▂▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  96.3 μs          Histogram: frequency by time          515 μs <

 Memory estimate: 400.08 KiB, allocs estimate: 4356.

In [17]:
# Creating the function to determine the length of the paths the neutrons take within the sample.
# Not broadcasted.

function path_len(E2s, E3s, D, Ps, dets, origin, vertices, indices)
    """
    Calculates the distance between a given point in the sample (origin) and a triangular face of the mesh that describes the surface. 
    Iterates through each face to determine which one is intersected.
    Exploits the method described in 'Fast, Minimum Storage Ray-Triangle Intersection' by Moller and Trumbore.

    Parameters
    ----------
    E2s (N_faces x 3 array of floats): Array containing vectors parallel to each face, equal to V2 - V1.
    E3s (N_faces x 3 array of floats): Array containing vectors parallel to each face, equal to V3 - V1.
    D (3-dimensional array of floats): Direction vector.
    Ps (N_faces x 3 array of floats): Array containing P = D x E3 for each face.
    dets (N_faces - dimensional array of floats): Array containing det = P.E2 = (D x E3).E2 for each face.
    origin (3-dimensional array of floats): Coordinates of scattering sites.
    vertices (N_faces x 3 array of floats): Vertices of triangular faces.
    indices (N_faces x 3 array of floats): Indices describing which vertices form which triangles.

    Returns
    -------
    path_length (float): Distance between origin and the surface the neutron path intersects, in units of the .stl file.
    """
    # If det = P.E2 = (D x E3).E2 = 0, the path is parallel to the triangular face, so it can never intersect it.
    # Keeping only positive determinants, equivalent to triangular faces in the forward direction.
    idx = findall(dets .> 1e-10)
    # Iterating through each valid face.
    for j in idx
        # Calculating T = origin - V1 and Q = T x E2 required for the MT algorithm.
        T = origin - vertices[indices[j][1]]
        Q = cross(T, E2s[j])
        # Calculating the barycentric coordinates, (u,v), of the intersection.
        u = (1 / dets[j]) * (dot(Ps[j], T))
        v = (1 / dets[j]) * (dot(Q, D))
        # Determining whether the intersection point lies within the triangle.
        if v ≥ 0 && u ≥ 0 && (u + v) ≤ 1
            t = (1 / dets[j]) * dot(Q, E3s[j])
            # Determining the path length based on t and the magnitude of the inputted direction vector.
            path_length = abs(t) * norm(D)
            return path_length
        end
    end
end
@benchmark path_len(E2s, E3s, -ki, cross.(fill(-ki, N_faces), E3s), dot.(Pi, E2s), MC_coords[1,:], vertices, indices)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   69.200 μs … 167.733 ms  ┊ GC (min … max):  0.00% … 99.88%
 Time  (median):     164.250 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   211.416 μs ±   2.035 ms  ┊ GC (mean ± σ):  17.66% ±  2.23%

  ▁█▁ ▁▇▃▁   ▁   ▁▄       ▅▅▂▂                                   
  ███▅████▆▆▆█▆▇▆██▆▆▅▇▅▅██████▇▅▅▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁ ▄
  69.2 μs          Histogram: frequency by time          416 μs <

 Memory estimate: 227.57 KiB, allocs estimate: 4309.

In [20]:
# Defining the function that will calculate the attenuation factor given a certain energy bin and wavevector.
# Broadcasted.

function Atten(ki, kf, axsi, Ef, vertices, indices, E2s, E3s, MC_coords, Lis)
    """
    Calculates the attenuation factor given a set initial and final energy and wavevector.

    Parameters
    ----------
    ki (3-dimensional array of floats): Pre-scattering wavevector of neutron, in Angstrom^-1.
    kf (3-dimensional array of floats): Post-scattering wavevector of neutron, in Angstrom^-1.
    axsi (float): Pre-scattering absorption cross section, in cm^2.
    Ef (float): Post-scattering energy of neutron, in meV.
    vertices (N_faces x 3 array of floats): Vertices of the triangular faces.
    indices (N_faces x 3 array of floats): Indices describing which vertices correspond to which triangles.
    E2s (N_faces x 3 array of floats): Array containing vectors parallel to each face, equal to V2 - V1.
    E3s (N_faces x 3 array of floats): Array containing vectors parallel to each face, equal to V3 - V1.
    MC_coords (N_MC x 3 array of floats): Coordinates of sample points used in MC method.
    Lis (N_MC - dimensional array of floats): Pre-scattering path length of neutron, in units of .stl file.

    Returns
    -------
    A (float): Attenuation factor.
    """
    # Calculating the absorption cross section after the neutron scatters.
    axsf = axs(Ef, axs_ref, E_ref)
    # Setting up the Moller-Trumbore algorithm for ray-triangle intersections.
    # Calculating cross products, P = D x E3, for the direction vector, Df = kf, and for each face.
    Dfs = Vector{Vector{Float32}}(undef, N_faces)
    Pf = cross.(fill!(Dfs, kf), E3s)
    # Calculating the determinant = P.E2 = (D x E3).E2 for the direction vector, Df, and for each face.
    detf = dot.(Pf, E2s)
    A = 0
    for i in 1:N_MC
        # Calculating the path length, Lf, at this sample point.
        Lf = path_len(E2s, E3s, kf, Pf, detf, MC_coords[i, :], vertices, indices)
        # Adding the attenuation factor contribution from this sample point to A.
        A += (1 / N_MC) * exp(-n * axsi * Lis[i]) * exp(-n * axsf * Lf)
    end
    return A
end
@benchmark Atten(ki, [kx[1,1], ky[1,1], kz[1,1]], Ei, Ef_bins[1], vertices, indices, E2s, E3s, MC_coords, Lis)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   68.600 μs … 148.105 ms  ┊ GC (min … max):  0.00% … 99.84%
 Time  (median):     151.950 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   187.904 μs ±   1.771 ms  ┊ GC (mean ± σ):  17.67% ±  2.22%

  ▆█           ▃       ▁    ▃▂                                   
  ██▅▃█▇▄▂▂▂▁▁▅█▄▃▄▅▄▂▄█▄▂▂▅██▇▅▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▃
  68.6 μs          Histogram: frequency by time          366 μs <

 Memory estimate: 218.73 KiB, allocs estimate: 4310.

In [19]:
# Defining the function that will calculate the attenuation factor given a certain energy bin and wavevector.
# Not broadcasted.

function Atten(ki, kf, Ei, Ef, vertices, indices, E2s, E3s, MC_coords, Lis)
    """
    Calculates the attenuation factor given a set initial and final energy and wavevector.

    Parameters
    ----------
    ki (3-dimensional array of floats): Pre-scattering wavevector of neutron, in Angstrom^-1.
    kf (3-dimensional array of floats): Post-scattering wavevector of neutron, in Angstrom^-1.
    Ei (float): Pre-scattering energy of neutron, in meV.
    Ef (float): Post-scattering energy of neutron, in meV.
    vertices (N_faces x 3 array of floats): Vertices of the triangular faces.
    indices (N_faces x 3 array of floats): Indices describing which vertices correspond to which triangles.
    E2s (N_faces x 3 array of floats): Array containing vectors parallel to each face, equal to V2 - V1.
    E3s (N_faces x 3 array of floats): Array containing vectors parallel to each face, equal to V3 - V1.
    MC_coords (N_MC x 3 array of floats): Coordinates of sample points used in MC method.
    Lis (N_MC - dimensional array of floats): Pre-scattering path length of neutron, in units of .stl file.

    Returns
    -------
    A (float): Attenuation factor.
    """
    # Calculating the absorption cross sections before and after the neutron scatters.
    axsi = axs(Ei, axs_ref, E_ref)
    axsf = axs(Ef, axs_ref, E_ref)
    # Setting up the Moller-Trumbore algorithm for ray-triangle intersections.
    # Direction vector for Lf calculation.
    Df = kf
    Pf = Vector(undef, N_faces)
    detf = zeros(N_faces)
    for j in 1:N_faces
        # Calculating cross products, P = D x E3, for the direction vector, Df, and for each face.
        Pf[j] = cross(Df, E3s[j])
        # Calculating the determinant = P.E2 = (D x E3).E2 for the direction vector, Df, and for each face.
        detf[j] = dot(Pf[j], E2s[j])
    end
    A = 0
    for i in 1:N_MC
        origin = MC_coords[i, :]
        # Calculating the path length, Lf, at this sample point.
        Lf = path_len(E2s, E3s, Df, Pf, detf, origin, vertices, indices)
        # Adding the attenuation factor contribution from this sample point to A.
        A += (1 / N_MC) * exp(-n * axsi * Lis[i]) * exp(-n * axsf * Lf)
    end
    return A
end
@benchmark Atten(ki, [kx[1,1], ky[1,1], kz[1,1]], Ei, Ef_bins[1], vertices, indices, E2s, E3s, MC_coords, Lis)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  215.000 μs … 154.167 ms  ┊ GC (min … max):  0.00% … 99.73%
 Time  (median):     356.150 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   410.263 μs ±   1.957 ms  ┊ GC (mean ± σ):  11.81% ±  3.67%

  ▇▅█             ▄                                              
  ███▅▃▂▂▂▂▃▃▃▂▃▅▃█▇▅▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  215 μs           Histogram: frequency by time          936 μs <

 Memory estimate: 361.94 KiB, allocs estimate: 11570.

In [15]:
# The pre-scattering neutron path lengths are dependent only on the MC coordinates.
# They can, therefore, be calculated and stored.

Lis = zeros(N_MC)
Pi = cross.(fill(-ki, N_faces), E3s)
deti = dot.(Pi, E2s)
for i in 1:N_MC
    Lis[i] = path_len(E2s, E3s, -ki, Pi, deti, MC_coords[i, :], vertices, indices)
end

In [21]:
# Defining the function that calculates the grid of attenuation factors.

function Atten_grid(data, kx, ky, kz, ki, Ei, Ef_bins, vertices, indices, E2s, E3s, MC_coords, Lis)
    """
    Calculates the attenuation factor for every non-zero, non-NaN, signal and stores in a grid of detector against energy bin.
    
    Parameters
    ----------
    data (N_bins x N_detectors array of floats): Neutron signal measured at different detectors for different energy bins.
    kx (N_bins x N_detectors array of floats): Post-scattering neutron wavevector component in x direction, in Angstrom^-1.
    ky (N_bins x N_detectors array of floats): Post-scattering neutron wavevector component in y direction, in Angstrom^-1.
    kz (N_bins x N_detectors array of floats): Post-scattering neutron wavevector component in z direction, in Angstrom^-1.
    ki (3-dimensional array of floats): Pre-scattering neutron wavevector, in Angstrom^-1.
    Ei (float): Pre-scattering neutron energy, in meV.
    Ef_bins (N_bins - dimensional array of floats): Post-scattering neutron energy of each bin, in meV.
    vertices (N_faces x 3 array of floats): Vertices of the triangular faces.
    indices (N_faces x 3 array of floats): Indices describing which vertices correspond to which triangles.
    E2s (N_faces x 3 array of floats): Array containing vectors parallel to each face, equal to V2 - V1.
    E3s (N_faces x 3 array of floats): Array containing vectors parallel to each face, equal to V3 - V1.
    MC_coords (N_MC x 3 array of floats): Coordinates of sample points used in MC method.
    Lis (N_MC - dimensional array of floats): Pre-scattering path length of neutron, in units of .stl file.

    Returns
    -------
    A_grid (N_bins x N_detectors array of floats): Attenuation factor grid.
    """
    A_grid = zeros(N_bins, N_detectors)
    # Determining the locations in which the signal is either NaN or 0 as we don't want to calculate A there.
    idx = findall(.~((data .== 0) .| (isnan.(data))))
    for I in idx
        kf = [kx[I], ky[I], kz[I]]
        A_grid[I] = Atten(ki, kf, Ei, Ef_bins[I[1]], vertices, indices, E2s, E3s, MC_coords, Lis)
    end
    return A_grid
end

@benchmark Atten_grid(data, kx, ky, kz, ki, Ei, Ef_bins, vertices, indices, E2s, E3s, MC_coords, Lis)

BenchmarkTools.Trial: 1 sample with 1 evaluation per sample.
 Single result which took 37.259 s (15.69% GC) to evaluate,
 with a memory estimate of 66.50 GiB, over 1368339955 allocations.

In [22]:
# Testing the time taken to output this grid of attenuation factors.

A_grid = Atten_grid(data, kx, ky, kz, ki, Ei, Ef_bins, vertices, indices, E2s, E3s, MC_coords, Lis)
display(A_grid)
display(A_grid[160,6])

320×98304 Matrix{Float64}:
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 ⋮                        ⋮              ⋱                 ⋮              
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 

0.0